In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📝 Resume Bullet Improver

## What We're Building

A tool that takes **weak resume bullet points** and rewrites them into
**impact-focused statements** following the STAR format:

- **S**ituation — What was the context?
- **T**ask — What was your responsibility?
- **A**ction — What did you do?
- **R**esult — What was the measurable outcome?

### Before vs After
```
❌ "Worked on the company website"
✅ "Redesigned company website using React, improving page load speed by 40%
    and increasing user engagement by 25% over 3 months"
```

## Key LangChain Concept: Prompt Templates

We'll use **prompt templates** with variables to create reusable, parameterized
prompts. This lets us change the tone, target role, and industry without
rewriting the entire prompt.

## Stack
- **LangChain** — prompt templates and chains
- **Ollama (qwen3.5:9b)** — local LLM

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama -q

from langchain_ollama import ChatOllama
from langchain.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")

## Step 2 — Initialize the LLM

In [ ]:
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0.5,  # Slightly creative for resume writing
)

print("LLM ready!")

## Step 3 — Build the Bullet Improver Prompt

We create a **parameterized prompt** that accepts:
- `bullet` — the original bullet point
- `target_role` — the job role we're targeting
- `tone` — professional, confident, technical, creative

### Why Prompt Templates?

Instead of hardcoding everything into a string, templates let us:
1. **Reuse** the same prompt structure with different inputs
2. **Swap variables** without touching prompt logic
3. **Validate** that all required variables are provided
4. **Compose** prompts from smaller building blocks

In [ ]:
# System message sets the assistant's persona
system_template = SystemMessagePromptTemplate.from_template(
    """You are an expert resume coach with 15 years of experience helping professionals
land jobs at top companies. You specialize in transforming vague, weak bullet points
into powerful, impact-focused statements.

Your principles:
- Start every bullet with a strong ACTION VERB (Led, Developed, Optimized, Spearheaded)
- Include QUANTIFIABLE RESULTS whenever possible (%, $, time saved, users impacted)
- Follow the STAR method: Situation, Task, Action, Result
- Keep each bullet to 1-2 lines maximum
- Tailor language to the target role and industry"""
)

# Human message has the actual task with variables
human_template = HumanMessagePromptTemplate.from_template(
    """Improve the following resume bullet point.

Original bullet: "{bullet}"
Target role: {target_role}
Desired tone: {tone}

Provide:
1. **Improved Version** — The rewritten bullet point
2. **Why It's Better** — Brief explanation of what changed
3. **Power Verbs Used** — List the strong action verbs
4. **Two Alternatives** — Two more variations for the same bullet"""
)

# Combine into a chat prompt
improve_prompt = ChatPromptTemplate.from_messages([
    system_template,
    human_template,
])

# Build the chain
improve_chain = improve_prompt | llm | StrOutputParser()

# Show the prompt variables
print(f"Prompt variables: {improve_prompt.input_variables}")

## Step 4 — Improve Some Bullet Points!

Let's test with real-world weak bullets that people commonly write.

In [ ]:
# Example 1: Vague software engineering bullet
result = improve_chain.invoke({
    "bullet": "Worked on the backend of the application",
    "target_role": "Senior Software Engineer",
    "tone": "technical and confident",
})

print("📝 BULLET IMPROVEMENT")
print("=" * 60)
print(f"Original: \"Worked on the backend of the application\"")
print(f"Target: Senior Software Engineer")
print("-" * 60)
print(result)

In [ ]:
# Example 2: Weak data science bullet
result = improve_chain.invoke({
    "bullet": "Used machine learning to analyze customer data",
    "target_role": "Data Scientist",
    "tone": "professional and results-oriented",
})

print("📝 BULLET IMPROVEMENT")
print("=" * 60)
print(f'Original: "Used machine learning to analyze customer data"')
print(f"Target: Data Scientist")
print("-" * 60)
print(result)

In [ ]:
# Example 3: Generic management bullet
result = improve_chain.invoke({
    "bullet": "Managed a team and delivered projects on time",
    "target_role": "Engineering Manager",
    "tone": "leadership-focused and strategic",
})

print("📝 BULLET IMPROVEMENT")
print("=" * 60)
print(f'Original: "Managed a team and delivered projects on time"')
print(f"Target: Engineering Manager")
print("-" * 60)
print(result)

## Step 5 — Batch Processing (Improve an Entire Resume Section)

In [ ]:
# A batch of weak bullets from a hypothetical resume
weak_bullets = [
    "Responsible for testing software",
    "Helped customers with their problems",
    "Made dashboards for the team",
    "Participated in code reviews",
    "Wrote documentation for APIs",
]

target_role = "Full Stack Developer"
tone = "technical and impactful"

# Batch prompt — improves all bullets at once (more efficient)
batch_prompt = ChatPromptTemplate.from_messages([
    system_template,
    HumanMessagePromptTemplate.from_template(
        """Improve ALL of the following resume bullet points for a {target_role} role.
Use a {tone} tone.

For each bullet, provide:
- Original
- Improved version (1-2 lines, starts with action verb, includes metrics)

Bullets to improve:
{bullets}"""
    ),
])

batch_chain = batch_prompt | llm | StrOutputParser()

bullets_text = "\n".join(f"{i+1}. {b}" for i, b in enumerate(weak_bullets))

print("🔄 Processing batch of 5 bullets...\n")
batch_result = batch_chain.invoke({
    "target_role": target_role,
    "tone": tone,
    "bullets": bullets_text,
})

print(batch_result)

## Step 6 — Tone Comparison

Let's see how the **same bullet** looks in different tones.
This demonstrates the power of prompt template variables.

In [ ]:
# Same bullet, different tones
bullet = "Built a mobile app for the company"
target_role = "Mobile Developer"

tones = [
    "highly technical (emphasize frameworks and architecture)",
    "business-focused (emphasize revenue and user growth)",
    "creative and startup-style (emphasize innovation and speed)",
]

# Simple single-output prompt for comparison
tone_prompt = ChatPromptTemplate.from_template(
    """You are an expert resume writer. Rewrite this resume bullet point.
Give ONLY the improved bullet (one line, no explanation).

Original: "{bullet}"
Target role: {target_role}
Tone: {tone}

Improved bullet:"""
)

tone_chain = tone_prompt | llm | StrOutputParser()

print(f'Original: "{bullet}"\n')
print("Same bullet in different tones:")
print("=" * 60)

for tone in tones:
    improved = tone_chain.invoke({
        "bullet": bullet,
        "target_role": target_role,
        "tone": tone,
    })
    print(f"\n🎯 {tone.split('(')[0].strip().upper()}")
    print(f"   {improved}")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **SystemMessagePromptTemplate** | Sets the AI's persona/role (resume coach) |
| **HumanMessagePromptTemplate** | Defines the user's request with variables |
| **ChatPromptTemplate** | Combines system + human messages into a chat prompt |
| **input_variables** | Template placeholders like `{bullet}`, `{tone}` |
| **Chain (LCEL)** | `prompt \| llm \| parser` composable pipeline |
| **Batch Processing** | Send multiple items in one prompt for efficiency |

## 🔧 Customization Ideas

- **Industry presets**: Add industry-specific jargon (finance, healthcare, tech)
- **ATS optimization**: Include keywords from a job description
- **Scoring**: Have the LLM rate the original vs improved bullet (1-10)
- **Full resume mode**: Process an entire resume and rewrite all sections